# 05 — Supervised Vulnerability Classification

This notebook tests whether the three vulnerability classes created in Notebook 04 can be reproduced from the underlying county-year indicators. It compares linear, tree-based, and boosted classifiers using an expanding-window temporal design.

**Input:** `data/processed/florida_county_year_vulnerability_2011_2025.csv`  
**Primary outputs:** model comparison, held-out predictions, coefficient and feature-importance tables, and the final Logistic Regression pipeline used by Notebook 08.

The 2023–2025 sample is a **later-period held-out evaluation**, not an independent external test set. Target labels were constructed from the full 2011–2025 panel in Notebook 04, so results describe class reproducibility rather than prospective forecasting.


## 1. Setup

Project paths and split years come from the shared configuration module. The original 25-trial Optuna search results are stored below as fixed parameters so routine reruns are deterministic and fast. Set `RUN_HYPERPARAMETER_SEARCH = True` only when intentionally repeating model selection.


In [12]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


def locate_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (  # noqa: E402
    RANDOM_STATE,
    STUDY_END_YEAR,
    STUDY_START_YEAR,
    TEST_START_YEAR,
    TRAIN_END_YEAR,
)

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "florida_county_year_vulnerability_2011_2025.csv"
)
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_MODELS = PROJECT_ROOT / "results" / "models"
RESULTS_TABLES.mkdir(parents=True, exist_ok=True)
RESULTS_MODELS.mkdir(parents=True, exist_ok=True)

RUN_HYPERPARAMETER_SEARCH = True
N_OPTUNA_TRIALS = 25

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_PATH}")
print(f"Repeat Optuna search: {RUN_HYPERPARAMETER_SEARCH}")
print(f"XGBoost version: {xgboost.__version__}")


Project root: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation
Input: C:\Users\saadm\OneDrive\Documents\MRP\climate-risk-housing-valuation\data\processed\florida_county_year_vulnerability_2011_2025.csv
Repeat Optuna search: True
XGBoost version: 3.2.0


## 2. Load and validate the modelling dataset

Notebook 04 is the sole upstream dependency. Validation checks protect the county-year panel, target coverage, and expected study period before any model is fitted.


In [13]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        "Run cleaned Notebook 04 first. Missing input: " f"{INPUT_PATH}"
    )

county_year_data = pd.read_csv(INPUT_PATH, dtype={"STCOFIPS": "string"})
county_year_data["STCOFIPS"] = county_year_data["STCOFIPS"].str.zfill(5)
county_year_data = county_year_data.sort_values(
    ["Year", "STCOFIPS"]
).reset_index(drop=True)

required_columns = {"Year", "STCOFIPS", "RegionName", "vulnerability_class"}
missing_required = required_columns.difference(county_year_data.columns)
if missing_required:
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

duplicate_count = int(county_year_data.duplicated(["STCOFIPS", "Year"]).sum())
class_counts = county_year_data["vulnerability_class"].value_counts()

assert duplicate_count == 0, "Duplicate county-year rows detected."
assert county_year_data["vulnerability_class"].notna().all()
assert set(class_counts.index) == {"Low", "Medium", "High"}
assert county_year_data["Year"].min() == STUDY_START_YEAR
assert county_year_data["Year"].max() == STUDY_END_YEAR

dataset_audit = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Columns",
            "Counties",
            "Year range",
            "Duplicate county-years",
            "Missing targets",
        ],
        "value": [
            len(county_year_data),
            county_year_data.shape[1],
            county_year_data["STCOFIPS"].nunique(),
            f"{county_year_data['Year'].min()}–{county_year_data['Year'].max()}",
            duplicate_count,
            int(county_year_data["vulnerability_class"].isna().sum()),
        ],
    }
)

display(dataset_audit)
display(class_counts.rename("observations").to_frame())


,check,value
0,Rows,1000
1,Columns,123
2,Counties,67
3,Year range,2011–2025
4,Duplicate county-years,0
5,Missing targets,0


,observations
vulnerability_class,
Low,334
Medium,333
High,333


## 3. Define the target and leakage-safe predictors

The final composite score, its eight component scores, and all score-construction helper columns are excluded. Identifiers and geographic labels remain available only for reporting. The remaining numeric variables form the 67-predictor feature matrix used in the original analysis.


In [14]:
TARGET_COLUMN = "vulnerability_class"

FINAL_SCORE_COLUMNS = [
    "vulnerability_class",
    "climate_housing_vulnerability_score",
]
SUBSCORE_COLUMNS = [
    "housing_pressure_score",
    "affordability_stress_score",
    "climate_exposure_score",
    "disaster_history_score",
    "socioeconomic_vulnerability_score",
    "resilience_adjustment_score",
    "insurance_loss_stress_score",
    "spatial_spillover_score",
]
METADATA_COLUMNS = [
    "STCOFIPS",
    "RegionID",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
    "COUNTY",
]

scaled_columns = [c for c in county_year_data if c.startswith("scaled_")]
log_columns = [c for c in county_year_data if c.startswith("log_")]
excluded_columns = list(
    dict.fromkeys(
        FINAL_SCORE_COLUMNS
        + SUBSCORE_COLUMNS
        + scaled_columns
        + log_columns
        + METADATA_COLUMNS
    )
)
excluded_columns = [c for c in excluded_columns if c in county_year_data.columns]

numeric_columns = county_year_data.select_dtypes(include=np.number).columns.tolist()
feature_columns = [c for c in numeric_columns if c not in excluded_columns]

assert TARGET_COLUMN not in feature_columns
assert "climate_housing_vulnerability_score" not in feature_columns
assert len(feature_columns) == 67, (
    f"Expected 67 predictors; found {len(feature_columns)}. "
    "Review upstream schema changes before modelling."
)

X = county_year_data[feature_columns].copy()
y = county_year_data[TARGET_COLUMN].copy()

feature_audit = pd.DataFrame(
    {
        "item": [
            "Available columns",
            "Excluded columns",
            "Final numeric predictors",
            "Predictors containing missing values",
        ],
        "count": [
            county_year_data.shape[1],
            len(excluded_columns),
            len(feature_columns),
            int(X.isna().any().sum()),
        ],
    }
)

display(feature_audit)
display(
    X.isna().sum().loc[lambda s: s.gt(0)].rename("missing_values").to_frame()
)


,item,count
0,Available columns,123
1,Excluded columns,56
2,Final numeric predictors,67
3,Predictors containing missing values,3


,missing_values
prev_year_housing_price,3
annual_price_growth_dollar,3
high_growth_flag,3


## 4. Temporal evaluation design

Models train on 2011–2022 and are evaluated on 2023–2025. Five expanding-window folds estimate performance stability without allowing later years into earlier validation periods. Median imputation is fitted inside every pipeline and therefore only learns from each training window.


In [15]:
train_mask = county_year_data["Year"].le(TRAIN_END_YEAR)
test_mask = county_year_data["Year"].ge(TEST_START_YEAR)

X_train, X_test = X.loc[train_mask], X.loc[test_mask]
y_train, y_test = y.loc[train_mask], y.loc[test_mask]

CV_YEAR_FOLDS = [
    {"fold": 1, "train_years": list(range(2011, 2015)), "validation_years": [2015, 2016]},
    {"fold": 2, "train_years": list(range(2011, 2017)), "validation_years": [2017, 2018]},
    {"fold": 3, "train_years": list(range(2011, 2019)), "validation_years": [2019, 2020]},
    {"fold": 4, "train_years": list(range(2011, 2021)), "validation_years": [2021]},
    {"fold": 5, "train_years": list(range(2011, 2022)), "validation_years": [2022]},
]

fold_rows = []
for fold in CV_YEAR_FOLDS:
    assert max(fold["train_years"]) < min(fold["validation_years"])
    fold_rows.append(
        {
            "fold": fold["fold"],
            "train_years": f"{min(fold['train_years'])}–{max(fold['train_years'])}",
            "validation_years": f"{min(fold['validation_years'])}–{max(fold['validation_years'])}",
            "train_rows": county_year_data["Year"].isin(fold["train_years"]).sum(),
            "validation_rows": county_year_data["Year"].isin(fold["validation_years"]).sum(),
        }
    )

split_summary = pd.concat(
    {
        "Training (2011–2022)": y_train.value_counts(),
        "Held out (2023–2025)": y_test.value_counts(),
    },
    axis=1,
).fillna(0).astype(int)

print(f"Training shape: {X_train.shape}")
print(f"Held-out shape: {X_test.shape}")
display(split_summary)
display(pd.DataFrame(fold_rows))


Training shape: (799, 67)
Held-out shape: (201, 67)


,Training (2011–2022),Held out (2023–2025)
vulnerability_class,,
Low,281,53
High,264,69
Medium,254,79


,fold,train_years,validation_years,train_rows,validation_rows
0,1,2011–2014,2015–2016,264,133
1,2,2011–2016,2017–2018,397,134
2,3,2011–2018,2019–2020,531,134
3,4,2011–2020,2021–2021,665,67
4,5,2011–2021,2022–2022,732,67


## 5. Evaluation helpers

All classifiers pass through the same temporal folds and report accuracy, macro-averaged precision/recall/F1, and weighted F1. XGBoost receives integer-encoded targets; all reported predictions are converted back to the original class labels.


In [16]:
label_encoder = LabelEncoder().fit(y_train)


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
    }


def fit_and_predict(estimator, X_fit, y_fit, X_predict, encode_target=False):
    fitted = clone(estimator)
    model_target = label_encoder.transform(y_fit) if encode_target else y_fit
    fitted.fit(X_fit, model_target)
    predictions = fitted.predict(X_predict)
    if encode_target:
        predictions = label_encoder.inverse_transform(predictions.astype(int))
    return fitted, predictions


def evaluate_temporally(model_name, estimator, encode_target=False):
    fold_results = []
    for fold in CV_YEAR_FOLDS:
        fold_train = county_year_data["Year"].isin(fold["train_years"])
        fold_valid = county_year_data["Year"].isin(fold["validation_years"])
        _, predictions = fit_and_predict(
            estimator,
            X.loc[fold_train],
            y.loc[fold_train],
            X.loc[fold_valid],
            encode_target,
        )
        metrics = classification_metrics(y.loc[fold_valid], predictions)
        metrics.update(
            {
                "model": model_name,
                "fold": fold["fold"],
                "train_years": f"{min(fold['train_years'])}-{max(fold['train_years'])}",
                "validation_years": f"{min(fold['validation_years'])}-{max(fold['validation_years'])}",
            }
        )
        fold_results.append(metrics)

    fitted, test_predictions = fit_and_predict(
        estimator, X_train, y_train, X_test, encode_target
    )
    test_metrics = classification_metrics(y_test, test_predictions)
    test_metrics["model"] = model_name

    prediction_table = county_year_data.loc[
        test_mask, ["Year", "STCOFIPS", "RegionName"]
    ].reset_index(drop=True)
    prediction_table["actual_vulnerability_class"] = y_test.reset_index(drop=True)
    prediction_table["predicted_vulnerability_class"] = test_predictions

    return {
        "cv": pd.DataFrame(fold_results),
        "test_metrics": test_metrics,
        "predictions": prediction_table,
        "fitted_model": fitted,
    }


def temporal_cv_macro_f1(estimator, encode_target=False):
    fold_scores = []
    for fold in CV_YEAR_FOLDS:
        fold_train = county_year_data["Year"].isin(fold["train_years"])
        fold_valid = county_year_data["Year"].isin(fold["validation_years"])
        _, predictions = fit_and_predict(
            estimator,
            X.loc[fold_train],
            y.loc[fold_train],
            X.loc[fold_valid],
            encode_target,
        )
        fold_scores.append(
            f1_score(
                y.loc[fold_valid], predictions, average="macro", zero_division=0
            )
        )
    return float(np.mean(fold_scores))


## 6. Model specifications and optional tuning

Only Random Forest and XGBoost were tuned in the original analysis. The fixed parameter dictionaries below are the winning configurations from the seeded 25-trial Optuna searches. This avoids silently changing the reported models during routine notebook reruns.


In [17]:
FROZEN_RF_PARAMS = {
    "n_estimators": 600,
    "max_depth": 8,
    "min_samples_split": 14,
    "min_samples_leaf": 5,
    "max_features": "log2",
}
FROZEN_XGB_PARAMS = {
    "n_estimators": 500,
    "max_depth": 2,
    "learning_rate": 0.14375148048694283,
    "subsample": 0.9647749916436704,
    "colsample_bytree": 0.9429178011056337,
    "min_child_weight": 7,
    "gamma": 0.6911407800788922,
    "reg_alpha": 0.007017110749133235,
    "reg_lambda": 0.028912644384523085,
}


def random_forest_pipeline(params):
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                RandomForestClassifier(
                    **params, random_state=RANDOM_STATE, n_jobs=-1
                ),
            ),
        ]
    )


def xgboost_pipeline(params):
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                XGBClassifier(
                    **params,
                    objective="multi:softmax",
                    eval_metric="mlogloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def rerun_optuna_search():
    try:
        import optuna
    except ImportError as exc:
        raise ImportError(
            "Install optuna or set RUN_HYPERPARAMETER_SEARCH = False."
        ) from exc

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def rf_objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
            "max_depth": trial.suggest_categorical(
                "max_depth", [4, 6, 8, 10, 12, None]
            ),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical(
                "max_features", ["sqrt", "log2", None]
            ),
        }
        return temporal_cv_macro_f1(random_forest_pipeline(params))

    def xgb_objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 7),
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.20, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.60, 1.00
            ),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 0.001, 10.0, log=True),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 0.001, 10.0, log=True
            ),
        }
        return temporal_cv_macro_f1(xgboost_pipeline(params), encode_target=True)

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    rf_study = optuna.create_study(direction="maximize", sampler=sampler)
    rf_study.optimize(rf_objective, n_trials=N_OPTUNA_TRIALS)

    xgb_study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    xgb_study.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS)
    return rf_study.best_params, xgb_study.best_params


if RUN_HYPERPARAMETER_SEARCH:
    tuned_rf_params, tuned_xgb_params = rerun_optuna_search()
else:
    tuned_rf_params = FROZEN_RF_PARAMS.copy()
    tuned_xgb_params = FROZEN_XGB_PARAMS.copy()

display(
    pd.DataFrame(
        {
            "model": ["Random Forest (Optuna)", "XGBoost (Optuna)"],
            "parameter_source": [
                "new Optuna search" if RUN_HYPERPARAMETER_SEARCH else "frozen original search",
                "new Optuna search" if RUN_HYPERPARAMETER_SEARCH else "frozen original search",
            ],
            "parameters": [tuned_rf_params, tuned_xgb_params],
        }
    )
)


c:\Users\saadm\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,parameter_source,parameters
0,Random Forest (Optuna),new Optuna search,"{'n_estimators': 600, 'max_depth': 8, 'min_sam..."
1,XGBoost (Optuna),new Optuna search,"{'n_estimators': 500, 'max_depth': 2, 'learnin..."


## 7. Fit and compare classifiers

The same split, folds, metrics, and preprocessing rules are applied to every specification. Logistic Regression and all tree models use their original settings, allowing direct comparison with the completed report.


In [18]:
MODEL_SPECS = {
    "Naive Baseline": (
        Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("classifier", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        False,
    ),
    "Logistic Regression": (
        Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                ),
            ]
        ),
        False,
    ),
    "Decision Tree": (
        Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "classifier",
                    DecisionTreeClassifier(
                        max_depth=5,
                        min_samples_leaf=10,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        False,
    ),
    "Random Forest": (
        random_forest_pipeline(
            {
                "n_estimators": 300,
                "max_depth": 8,
                "min_samples_leaf": 5,
                "min_samples_split": 2,
                "max_features": "sqrt",
            }
        ),
        False,
    ),
    "XGBoost": (
        xgboost_pipeline(
            {
                "n_estimators": 300,
                "max_depth": 4,
                "learning_rate": 0.05,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
            }
        ),
        True,
    ),
    "Random Forest (Optuna)": (
        random_forest_pipeline(tuned_rf_params),
        False,
    ),
    "XGBoost (Optuna)": (
        xgboost_pipeline(tuned_xgb_params),
        True,
    ),
}

model_outputs = {}
for model_name, (estimator, encode_target) in MODEL_SPECS.items():
    print(f"Evaluating {model_name}...")
    model_outputs[model_name] = evaluate_temporally(
        model_name, estimator, encode_target
    )

all_cv_results = pd.concat(
    [output["cv"] for output in model_outputs.values()], ignore_index=True
)
cv_summary = (
    all_cv_results.groupby("model")
    .agg(
        cv_accuracy_mean=("accuracy", "mean"),
        cv_accuracy_std=("accuracy", "std"),
        cv_macro_f1_mean=("macro_f1", "mean"),
        cv_macro_f1_std=("macro_f1", "std"),
        cv_weighted_f1_mean=("weighted_f1", "mean"),
        cv_weighted_f1_std=("weighted_f1", "std"),
    )
    .reset_index()
)

test_summary = pd.DataFrame(
    [output["test_metrics"] for output in model_outputs.values()]
).rename(
    columns={
        "accuracy": "test_accuracy",
        "macro_precision": "test_macro_precision",
        "macro_recall": "test_macro_recall",
        "macro_f1": "test_macro_f1",
        "weighted_f1": "test_weighted_f1",
    }
)

model_comparison = cv_summary.merge(test_summary, on="model", how="left")
metric_columns = model_comparison.select_dtypes(include=np.number).columns
model_comparison[metric_columns] = model_comparison[metric_columns].round(4)
model_comparison = model_comparison.sort_values(
    "test_macro_f1", ascending=False
).reset_index(drop=True)

display(model_comparison)


Evaluating Naive Baseline...
Evaluating Logistic Regression...
Evaluating Decision Tree...
Evaluating Random Forest...
Evaluating XGBoost...
Evaluating Random Forest (Optuna)...
Evaluating XGBoost (Optuna)...


,model,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,cv_weighted_f1_mean,cv_weighted_f1_std,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1
0,Logistic Regression,0.7758,0.0922,0.7345,0.1097,0.7639,0.1070,0.7761,0.8038,0.8077,0.7687,0.7622
1,XGBoost (Optuna),0.8117,0.0403,0.7934,0.0877,0.8198,0.0422,0.7313,0.7438,0.7673,0.7218,0.7077
2,Random Forest,0.7669,0.0431,0.7371,0.0568,0.7760,0.0500,0.6418,0.6445,0.6708,0.6321,0.6159
3,XGBoost,0.7938,0.0402,0.7591,0.0725,0.8041,0.0410,0.6468,0.6543,0.6887,0.6221,0.6033
4,Random Forest (Optuna),0.7713,0.0357,0.7469,0.0606,0.7801,0.0433,0.6169,0.6260,0.6448,0.6029,0.5845
5,Decision Tree,0.6397,0.0247,0.5655,0.1091,0.6596,0.0512,0.5473,0.5575,0.5787,0.5295,0.5175
6,Naive Baseline,0.1782,0.1533,0.0932,0.0735,0.0767,0.0893,0.2637,0.0879,0.3333,0.1391,0.1100


## 8. Inspect the selected model

To match the completed analysis, the final classifier is the model with the highest macro F1 on the 2023–2025 held-out period. Because this period is also used to compare candidates, the resulting value should be reported as a later-period evaluation—not as an untouched final test estimate.


In [19]:
best_model_name = model_comparison.loc[0, "model"]
best_output = model_outputs[best_model_name]
best_predictions = best_output["predictions"].copy()

CLASS_ORDER = ["Low", "Medium", "High"]
best_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        best_predictions["actual_vulnerability_class"],
        best_predictions["predicted_vulnerability_class"],
        labels=CLASS_ORDER,
    ),
    index=[f"Actual {label}" for label in CLASS_ORDER],
    columns=[f"Predicted {label}" for label in CLASS_ORDER],
)
best_classification_report = pd.DataFrame(
    classification_report(
        best_predictions["actual_vulnerability_class"],
        best_predictions["predicted_vulnerability_class"],
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
).T

print(f"Selected model: {best_model_name}")
display(best_confusion_matrix)
display(best_classification_report.round(4))


Selected model: Logistic Regression


,Predicted Low,Predicted Medium,Predicted High
Actual Low,53,0,0
Actual Medium,33,38,8
Actual High,0,4,65


,precision,recall,f1-score,support
Low,0.6163,1.0000,0.7626,53.0000
Medium,0.9048,0.4810,0.6281,79.0000
High,0.8904,0.9420,0.9155,69.0000
accuracy,0.7761,0.7761,0.7761,0.7761
macro avg,0.8038,0.8077,0.7687,201.0000
weighted avg,0.8238,0.7761,0.7622,201.0000


## 9. Model interpretation tables

Logistic coefficients are expressed per one-standard-deviation increase because scaling occurs inside the pipeline. Their signs are class-specific associations, not causal effects. Tree importances describe predictive usage and likewise should not be interpreted causally.


In [20]:
logistic_pipeline = model_outputs["Logistic Regression"]["fitted_model"]
logistic_classifier = logistic_pipeline.named_steps["classifier"]

coefficient_rows = []
for class_label, coefficients in zip(
    logistic_classifier.classes_, logistic_classifier.coef_
):
    for feature, coefficient in zip(feature_columns, coefficients):
        coefficient_rows.append(
            {
                "class": class_label,
                "feature": feature,
                "coefficient": coefficient,
                "absolute_coefficient": abs(coefficient),
                "direction": (
                    "Positive"
                    if coefficient > 0
                    else "Negative"
                    if coefficient < 0
                    else "Zero"
                ),
            }
        )

logistic_coefficients = pd.DataFrame(coefficient_rows)
logistic_coefficients[["coefficient", "absolute_coefficient"]] = (
    logistic_coefficients[["coefficient", "absolute_coefficient"]].round(4)
)

tree_importance_tables = []
for model_name in ["Decision Tree", "Random Forest", "XGBoost"]:
    classifier = model_outputs[model_name]["fitted_model"].named_steps["classifier"]
    tree_importance_tables.append(
        pd.DataFrame(
            {
                "model": model_name,
                "feature": feature_columns,
                "importance": classifier.feature_importances_,
            }
        )
    )

tree_feature_importance = (
    pd.concat(tree_importance_tables, ignore_index=True)
    .sort_values(["model", "importance"], ascending=[True, False])
    .reset_index(drop=True)
)

high_coefficients = logistic_coefficients.query("`class` == 'High'")
top_high = pd.concat(
    [
        high_coefficients.nlargest(10, "coefficient").assign(ranking="Positive"),
        high_coefficients.nsmallest(10, "coefficient").assign(ranking="Negative"),
    ],
    ignore_index=True,
)

display(top_high[["ranking", "feature", "coefficient"]])
display(tree_feature_importance.groupby("model", group_keys=False).head(10))


,ranking,feature,coefficient
0,Positive,CFLD_RISKS,2.5871
1,Positive,neighbor_high_growth_share,1.8181
2,Positive,nfip_avg_claim_payment,1.3848
3,Positive,nfip_claim_year_indicator,1.1128
4,Positive,neighbor_high_volatility_share,1.1041
5,Positive,HRCN_RISKS,1.0960
6,Positive,SOVI_SCORE,0.9710
7,Positive,price_to_income_ratio,0.9082
8,Positive,hurricane_disaster_count,0.8106
9,Positive,neighbor_avg_price_growth_pct,0.7850


,model,feature,importance
0,Decision Tree,CFLD_RISKS,0.2831
1,Decision Tree,neighbor_avg_price_growth_pct,0.2805
2,Decision Tree,RESL_SCORE,0.0999
3,Decision Tree,climate_disaster_count,0.0615
4,Decision Tree,annual_price_growth_dollar,0.0444
5,Decision Tree,nfip_recent_3yr_claim_payment,0.0418
6,Decision Tree,SOVI_SCORE,0.0411
7,Decision Tree,neighbor_avg_housing_price,0.0383
8,Decision Tree,nfip_cumulative_claim_payment,0.0374
9,Decision Tree,nfip_avg_claim_payment,0.0251


## 10. Export results and the explainability pipeline

The final Logistic Regression pipeline, its ordered feature list, and class-probability predictions are saved under the filenames expected by Notebook 08. Fold-level metrics and selected-model diagnostics are also exported to make the evaluation auditable without reopening the notebook.


In [21]:
final_logistic_pipeline = model_outputs["Logistic Regression"]["fitted_model"]
logistic_predictions = final_logistic_pipeline.predict(X_test)
logistic_probabilities = final_logistic_pipeline.predict_proba(X_test)
logistic_class_order = [
    str(c) for c in final_logistic_pipeline.named_steps["classifier"].classes_
]

logistic_probability_table = county_year_data.loc[
    test_mask, ["Year", "STCOFIPS", "RegionName"]
].reset_index(drop=True)
logistic_probability_table["actual_vulnerability_class"] = y_test.reset_index(
    drop=True
).astype(str)
logistic_probability_table["predicted_vulnerability_class"] = logistic_predictions

for class_index, class_label in enumerate(logistic_class_order):
    logistic_probability_table[f"probability_{class_label.lower()}"] = (
        logistic_probabilities[:, class_index]
    )

logistic_probability_table["predicted_probability"] = logistic_probabilities.max(
    axis=1
)
sorted_probabilities = np.sort(logistic_probabilities, axis=1)
logistic_probability_table["probability_margin"] = (
    sorted_probabilities[:, -1] - sorted_probabilities[:, -2]
)
logistic_probability_table["prediction_correct"] = (
    logistic_probability_table["actual_vulnerability_class"]
    == logistic_probability_table["predicted_vulnerability_class"]
)

output_paths = {
    "model_comparison": RESULTS_TABLES
    / "vulnerability_classification_model_comparison.csv",
    "cv_fold_results": RESULTS_TABLES
    / "vulnerability_classification_fold_results.csv",
    "best_predictions": RESULTS_TABLES
    / "best_model_vulnerability_classification_predictions.csv",
    "classification_report": RESULTS_TABLES
    / "vulnerability_classification_report.csv",
    "confusion_matrix": RESULTS_TABLES
    / "vulnerability_classification_confusion_matrix.csv",
    "logistic_coefficients": RESULTS_TABLES
    / "logistic_regression_feature_coefficients.csv",
    "tree_importance": RESULTS_TABLES / "tree_based_feature_importance.csv",
    "logistic_probabilities": RESULTS_TABLES
    / "logistic_regression_test_predictions_with_probabilities.csv",
    "pipeline": RESULTS_MODELS
    / "final_logistic_regression_vulnerability_pipeline.joblib",
    "feature_list": RESULTS_MODELS
    / "final_logistic_regression_feature_list.json",
    "metadata": RESULTS_MODELS / "final_logistic_regression_metadata.json",
}

model_comparison.to_csv(output_paths["model_comparison"], index=False)
all_cv_results.to_csv(output_paths["cv_fold_results"], index=False)
best_predictions.to_csv(output_paths["best_predictions"], index=False)
best_classification_report.to_csv(output_paths["classification_report"])
best_confusion_matrix.to_csv(output_paths["confusion_matrix"])
logistic_coefficients.to_csv(output_paths["logistic_coefficients"], index=False)
tree_feature_importance.to_csv(output_paths["tree_importance"], index=False)
logistic_probability_table.to_csv(
    output_paths["logistic_probabilities"], index=False
)
joblib.dump(final_logistic_pipeline, output_paths["pipeline"])

feature_list_export = {
    "feature_count": len(feature_columns),
    "features": feature_columns,
}
output_paths["feature_list"].write_text(
    json.dumps(feature_list_export, indent=2), encoding="utf-8"
)

metadata_export = {
    "model_name": "Logistic Regression",
    "selection_basis": "highest 2023-2025 held-out macro F1",
    "target_column": TARGET_COLUMN,
    "class_order": logistic_class_order,
    "feature_count": len(feature_columns),
    "training_year_start": int(county_year_data.loc[train_mask, "Year"].min()),
    "training_year_end": int(county_year_data.loc[train_mask, "Year"].max()),
    "testing_year_start": int(county_year_data.loc[test_mask, "Year"].min()),
    "testing_year_end": int(county_year_data.loc[test_mask, "Year"].max()),
    "train_end_year_setting": TRAIN_END_YEAR,
    "test_start_year_setting": TEST_START_YEAR,
    "random_state": RANDOM_STATE,
    "training_observations": len(X_train),
    "testing_observations": len(X_test),
    "cross_validation_folds": CV_YEAR_FOLDS,
    "hyperparameter_search_rerun": RUN_HYPERPARAMETER_SEARCH,
    "tuned_random_forest_parameters": tuned_rf_params,
    "tuned_xgboost_parameters": tuned_xgb_params,
    "scikit_learn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
}
output_paths["metadata"].write_text(
    json.dumps(metadata_export, indent=2), encoding="utf-8"
)

export_summary = pd.DataFrame(
    {
        "artifact": output_paths.keys(),
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in output_paths.values()],
    }
)

print(f"Selected model: {best_model_name}")
print(f"Saved {len(output_paths)} reproducible outputs.")
display(export_summary)


Selected model: Logistic Regression
Saved 11 reproducible outputs.


,artifact,path
0,model_comparison,results\tables\vulnerability_classification_mo...
1,cv_fold_results,results\tables\vulnerability_classification_fo...
2,best_predictions,results\tables\best_model_vulnerability_classi...
3,classification_report,results\tables\vulnerability_classification_re...
4,confusion_matrix,results\tables\vulnerability_classification_co...
5,logistic_coefficients,results\tables\logistic_regression_feature_coe...
6,tree_importance,results\tables\tree_based_feature_importance.csv
7,logistic_probabilities,results\tables\logistic_regression_test_predic...
8,pipeline,results\models\final_logistic_regression_vulne...
9,feature_list,results\models\final_logistic_regression_featu...


## 11. Key result

Logistic Regression provides the strongest later-period macro F1 in this comparison. Its performance—especially perfect recall for the Low class and strong recall for the High class—shows that the constructed vulnerability labels are largely recoverable from the underlying indicators. The Medium class remains the most difficult to identify, consistent with its position between the two more distinct extremes.

These results establish the supervised benchmark used in Notebook 06 and supply the fitted pipeline and coefficients used for explainability in Notebook 08.
